
# ⚠️ 컴플라이언스 승인 대기 — 운영진 답변 전까지 절대 제출하지 마세요 ⚠️

## Pseudo-labeling 멀티시드 검증 결과 (참고)
3시드 전부 일관되게 양수 — 진짜 신호로 판단됨:

| 시드 | baseline | +pseudo | 차이 |
|---|---|---|---|
| 42 | 0.74047 | 0.74076 | +0.00029 |
| 1004 | 0.74031 | 0.74048 | +0.00017 |
| 7 | 0.74031 | 0.74067 | +0.00036 |
| **평균** | 0.74036 | 0.74063 | **+0.00027** |

이 노트북은 그 검증된 설정으로 **test 예측까지 생성**합니다 (baseline은 이미 검증 끝났으므로
재실행 안 함 — pseudo 버전 3시드만 재실행해서 시간 절약).

## ⚠️ 제출 전 필수 확인
- 대회 운영진에게 pseudo-labeling 허용 여부를 문의해두었음
- **"허용" 답변을 받기 전까지 이 노트북이 만드는 CSV는 제출하지 않습니다**
- 파일명에도 `PENDING_APPROVAL`을 표시해서 다른 제출 파일과 혼동되지 않도록 함

---
**저장 위치(참고용, 제출 폴더 아님):** `experiment_history/2차/2차_pseudolabel_predict_PENDING_APPROVAL.ipynb`


In [1]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import warnings

warnings.filterwarnings("ignore")


In [2]:
DATA_DIR = Path("../../data")
SHARED_DIR = Path("..")
BLEND_DIR = SHARED_DIR / "blend_cache"
PENDING_DIR = Path("../pending_compliance_review")  # 제출 폴더와 분리된 별도 위치
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
TARGET_COL = "임신 성공 여부"
DEAD_COLS = ["불임 원인 - 여성 요인", "난자 채취 경과일"]
N_SPLITS = 5

SEEDS = [42, 1004, 7]  # 검증된 시드 그대로 재사용

PSEUDO_TOP_PCT = 0.03
PSEUDO_BOTTOM_PCT = 0.03

EMBRYO_COUNT_COLS = ["총 생성 배아 수","미세주입된 난자 수","미세주입에서 생성된 배아 수","이식된 배아 수",
    "미세주입 배아 이식 수","저장된 배아 수","미세주입 후 저장된 배아 수","해동된 배아 수",
    "해동 난자 수","수집된 신선 난자 수","저장된 신선 난자 수","혼합된 난자 수",
    "파트너 정자와 혼합된 난자 수","기증자 정자와 혼합된 난자 수"]
EMBRYO_BINARY_COLS = ["단일 배아 이식 여부","착상 전 유전 진단 사용 여부","동결 배아 사용 여부",
    "신선 배아 사용 여부","기증 배아 사용 여부","대리모 여부"]
CLIP_RULES = {"총 생성 배아 수":40,"수집된 신선 난자 수":40,"미세주입된 난자 수":45,
    "혼합된 난자 수":40,"저장된 배아 수":30,"배아 이식 경과일":7,"난자 혼합 경과일":7,
    "배아 해동 경과일":2,"난자 해동 경과일":1}
RATIO_SPECS = [
    ("총 생성 배아 수", "혼합된 난자 수", "fertilization_rate"),
    ("미세주입에서 생성된 배아 수", "미세주입된 난자 수", "icsi_fertilization_rate"),
    ("이식된 배아 수", "총 생성 배아 수", "embryo_utilization_rate"),
    ("저장된 배아 수", "총 생성 배아 수", "embryo_freezing_rate"),
    ("혼합된 난자 수", "수집된 신선 난자 수", "oocyte_utilization_rate"),
    ("이식된 배아 수", "해동된 배아 수", "thawed_embryo_transfer_ratio"),
]


## 1. 풍부한 피처셋 빌드 (donor_cat과 동일, train+test 둘 다)

In [3]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw_full = pd.read_csv(TEST_PATH)
test_ids = test_raw_full["ID"]
y = train_raw[TARGET_COL].values.astype(int)


def safe_ratio(df, numerator, denominator, new_col):
    df = df.copy()
    can = df[numerator].notna() & df[denominator].notna() & (df[denominator] > 0)
    df[f"{new_col}_available"] = can.astype(int)
    df[new_col] = np.where(can, df[numerator] / df[denominator], np.nan)
    return df


def build_full_features(df):
    df = df.copy()
    df = df.drop(columns=[c for c in DEAD_COLS if c in df.columns])
    df["is_DI"] = (df["시술 유형"] == "DI").astype(int)
    df["froze_embryo"] = df["동결 배아 사용 여부"].fillna(0).astype(int)

    embryo_cols_present = [c for c in (EMBRYO_COUNT_COLS + EMBRYO_BINARY_COLS) if c in df.columns]
    for col in embryo_cols_present:
        df[f"{col}_missing"] = df[col].isna().astype(int)
        is_di_missing = (df["시술 유형"] == "DI") & df[col].isna()
        df[f"{col}_not_applicable_DI"] = is_di_missing.astype(int)

    for col, upper in CLIP_RULES.items():
        if col in df.columns:
            df[f"{col}_high_flag"] = (df[col] > upper).astype(int)
            df[col] = df[col].clip(upper=upper)

    if "배아 이식 경과일" in df.columns:
        df["transfer_day_0_1_flag"] = df["배아 이식 경과일"].isin([0, 1]).astype(int)
        df["transfer_day_3_flag"] = (df["배아 이식 경과일"] == 3).astype(int)
        df["transfer_day_5_or_more_flag"] = (df["배아 이식 경과일"] >= 5).astype(int)

    for num, den, new in RATIO_SPECS:
        if num in df.columns and den in df.columns:
            df = safe_ratio(df, num, den, new)

    patient_mid = {"만18-34세":31,"만35-37세":36,"만38-39세":38.5,"만40-42세":41,
                    "만43-44세":43.5,"만45-50세":47.5,"알 수 없음":np.nan}
    donor_mid = {"만20세 이하":20,"만21-25세":23,"만26-30세":28,"만31-35세":33,
                 "만36-40세":38,"만41-45세":43,"알 수 없음":np.nan}
    if "난자 출처" in df.columns and "시술 당시 나이" in df.columns:
        df["patient_age_mid"] = df["시술 당시 나이"].map(patient_mid)
        donor_age_mid = df["난자 기증자 나이"].map(donor_mid) if "난자 기증자 나이" in df.columns else pd.Series(np.nan, index=df.index)
        donor_known = (df["난자 출처"] == "기증 제공") & donor_age_mid.notna()
        df["effective_maternal_age_mid"] = df["patient_age_mid"]
        df.loc[donor_known, "effective_maternal_age_mid"] = donor_age_mid[donor_known]
        df["donor_rejuvenation_gap"] = 0.0
        df.loc[donor_known, "donor_rejuvenation_gap"] = (
            df.loc[donor_known, "patient_age_mid"] - donor_age_mid[donor_known]
        )

    if "특정 시술 유형" in df.columns:
        s = df["특정 시술 유형"].astype(str)
        for token in ["IVF", "ICSI", "IUI", "ICI", "GIFT", "FER", "Generic DI", "IVI", "BLASTOCYST", "AH"]:
            safe = token.lower().replace(" ", "_")
            df[f"spec_has_{safe}"] = s.str.contains(token, regex=False, na=False).astype(int)

    if {"시술 당시 나이", "난자 출처"}.issubset(df.columns):
        df["age_oocyte_source"] = df["시술 당시 나이"].astype(str) + "_" + df["난자 출처"].astype(str)

    return df


train_feat = build_full_features(train_raw.drop(columns=["ID"]))
test_feat = build_full_features(test_raw_full.drop(columns=["ID"]))

X_train_full = train_feat.drop(columns=[TARGET_COL])
X_test_full = test_feat.copy()

SENTINEL = "정보없음"
obj_cols = X_train_full.select_dtypes(include=["object"]).columns.tolist()
for col in obj_cols:
    X_train_full[col] = X_train_full[col].fillna(SENTINEL)
    X_test_full[col] = X_test_full[col].fillna(SENTINEL)
    train_categories = sorted(set(X_train_full[col].astype(str).unique()) | {SENTINEL})
    X_train_full[col] = pd.Categorical(X_train_full[col].astype(str), categories=train_categories)
    X_test_full[col] = pd.Categorical(X_test_full[col].astype(str), categories=train_categories)
    X_test_full[col] = X_test_full[col].fillna(SENTINEL)
X_test_full = X_test_full[X_train_full.columns]

print(f"풍부한 피처셋: train {X_train_full.shape}, test {X_test_full.shape}")


풍부한 피처셋: train (256351, 145), test (90067, 145)


## 2. 가짜 라벨 선정 (검증 때와 동일)

In [4]:
ECDF_TEST_PATH = BLEND_DIR / "ecdf_stack_test.npy"
DONOR_TEST_PATH = BLEND_DIR / "test_donor_cat.npy"

if ECDF_TEST_PATH.exists():
    source_test_pred = np.load(ECDF_TEST_PATH)
    print("가짜 라벨 소스: ecdf_stack_test.npy")
elif DONOR_TEST_PATH.exists():
    source_test_pred = np.load(DONOR_TEST_PATH)
    print("가짜 라벨 소스: test_donor_cat.npy")
else:
    raise FileNotFoundError("기존 test 예측 파일을 찾을 수 없습니다")

top_thresh = np.quantile(source_test_pred, 1 - PSEUDO_TOP_PCT)
bottom_thresh = np.quantile(source_test_pred, PSEUDO_BOTTOM_PCT)
pseudo_pos_mask = source_test_pred >= top_thresh
pseudo_neg_mask = source_test_pred <= bottom_thresh

pseudo_idx = np.where(pseudo_pos_mask | pseudo_neg_mask)[0]
pseudo_y = np.where(pseudo_pos_mask[pseudo_idx], 1, 0)
X_pseudo = X_test_full.iloc[pseudo_idx].reset_index(drop=True)

print(f"가짜 라벨 행 수: {len(pseudo_idx)} (train 대비 {len(pseudo_idx)/len(y):.2%})")


가짜 라벨 소스: ecdf_stack_test.npy
가짜 라벨 행 수: 5404 (train 대비 2.11%)


## 3. Pseudo-label 버전 3시드 재학습 (이번엔 test 예측도 생성)

In [5]:
def run_catboost_pseudo_with_test(X, y, X_test, X_extra, y_extra, seed=42, n_splits=N_SPLITS):
    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    cat_idx = [X.columns.get_loc(c) for c in cat_cols]
    params = dict(loss_function="Logloss", eval_metric="AUC", iterations=1500, learning_rate=0.03,
                  depth=6, l2_leaf_reg=5, random_seed=seed, early_stopping_rounds=100,
                  allow_writing_files=False, verbose=False)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(y))
    test_pred = np.zeros(len(X_test))
    for tr_idx, val_idx in skf.split(X, y):
        X_tr, y_tr = X.iloc[tr_idx], y[tr_idx]
        X_va, y_va = X.iloc[val_idx], y[val_idx]

        X_tr = pd.concat([X_tr, X_extra], axis=0, ignore_index=True)
        y_tr = np.concatenate([y_tr, y_extra])

        tp = Pool(X_tr, y_tr, cat_features=cat_idx)
        vp = Pool(X_va, y_va, cat_features=cat_idx)
        m = CatBoostClassifier(**params)
        m.fit(tp, eval_set=vp, use_best_model=True)
        oof[val_idx] = m.predict_proba(vp)[:, 1]
        test_pred += m.predict_proba(Pool(X_test, cat_features=cat_idx))[:, 1] / n_splits
    return oof, test_pred


oof_per_seed, test_per_seed, aucs = [], [], []
start = time.time()
for seed in SEEDS:
    oof_s, test_s = run_catboost_pseudo_with_test(
        X_train_full, y, X_test_full, X_pseudo, pseudo_y, seed=seed
    )
    auc_s = roc_auc_score(y, oof_s)
    oof_per_seed.append(oof_s)
    test_per_seed.append(test_s)
    aucs.append(auc_s)
    print(f"[seed={seed}] AUC={auc_s:.5f}  ({time.time()-start:.0f}s 누적)")

oof_bagged = np.mean(oof_per_seed, axis=0)
test_bagged = np.mean(test_per_seed, axis=0)
bagged_auc = roc_auc_score(y, oof_bagged)

print(f"\n시드별 AUC: {[round(a, 5) for a in aucs]}")
print(f"3시드 배깅 OOF AUC: {bagged_auc:.5f}")
print(f"(참고: baseline 3시드 배깅은 약 0.74036 — 검증 단계에서 확인된 값)")


[seed=42] AUC=0.74076  (743s 누적)
[seed=1004] AUC=0.74048  (1429s 누적)
[seed=7] AUC=0.74067  (2164s 누적)

시드별 AUC: [0.74076, 0.74048, 0.74067]
3시드 배깅 OOF AUC: 0.74088
(참고: baseline 3시드 배깅은 약 0.74036 — 검증 단계에서 확인된 값)


## 4. 결과 저장 (제출 폴더 아닌 별도 위치 — 승인 대기)

In [6]:
BLEND_DIR.mkdir(exist_ok=True)
np.save(BLEND_DIR / "oof_pseudolabel_PENDING_APPROVAL.npy", oof_bagged)
np.save(BLEND_DIR / "test_pseudolabel_PENDING_APPROVAL.npy", test_bagged)

PENDING_DIR.mkdir(exist_ok=True)
submission = pd.DataFrame({"ID": test_ids, "probability": test_bagged})
out_path = PENDING_DIR / f"2차_pseudolabel_local{bagged_auc:.5f}_PENDING_APPROVAL.csv"
submission.to_csv(out_path, index=False)

print(f"저장 완료: {out_path}")
print()
print("="*60)
print("⚠️  이 파일은 운영진 승인 전까지 제출하지 마세요")
print("="*60)


저장 완료: ../pending_compliance_review/2차_pseudolabel_local0.74088_PENDING_APPROVAL.csv

⚠️  이 파일은 운영진 승인 전까지 제출하지 마세요


## 5. 다음 단계

- 운영진이 **허용**으로 답하면: 이 OOF/test를 `blend_cache`에 13번째 후보로 추가해서
  `2차_final_push_stacking.ipynb`의 ECDF 스태킹을 다시 돌려 최종 조합 재탐색
- 운영진이 **불허**로 답하면: 이 파일들은 삭제하고, 보고서에 "탐구했으나 규칙상 미채택"으로 기록
- 답변이 마감 전까지 안 오면: **보수적으로 미채택 처리** (애매하면 안 하는 원칙)
